In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Sampler 빠른 시작

Sampler의 핵심 작업은 하나 이상의 양자 Circuit 실행에서 출력 레지스터를 샘플링하는 것입니다. [동적 Circuit](/guides/execute-dynamic-circuits)과 매개변수화된 Circuit을 입력으로 허용합니다(매개변수화된 Circuit이 제출되는 경우 매개변수 값도 제공해야 합니다). Sampler는 또한 [오류 억제](/guides/error-mitigation-and-suppression-techniques)를 위한 내장 동적 분리 및 트월링을 지원합니다.

이 주제의 단계는 Sampler를 설정하고, 구성에 사용할 수 있는 옵션을 탐색하고, 프로그램에서 호출하는 방법을 설명합니다.


<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

## Sampler Primitive 사용 단계
### 1. 계정 초기화
Qiskit Runtime은 관리형 서비스이므로 먼저 계정을 초기화해야 합니다. 그런 다음 기댓값을 계산하는 데 사용할 QPU를 선택할 수 있습니다.

계정이 아직 설정되지 않은 경우 [IBM Cloud 계정 설정](/guides/cloud-setup) 주제의 단계를 따르세요.

:::note[분수 Gate]

새로 지원되는 [분수 Gate](/guides/fractional-gates)를 사용하려면 `QiskitRuntimeService` 인스턴스에서 백엔드를 요청할 때 `use_fractional_gates=True`를 설정하세요. 예를 들어:

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

이는 실험적 기능으로 향후 변경될 수 있습니다.

:::

In [2]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(127, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

### 2. Circuit 생성
Sampler Primitive에 대한 입력으로 최소 하나의 Circuit이 필요합니다.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 3036), ('sx', 1769), ('cz', 378), ('measure', 127), ('barrier', 1)])


Circuit과 Observable은 QPU에서 지원되는 명령만 사용하도록 변환되어야 합니다(*명령 집합 아키텍처(ISA)* Circuit이라고 함). 이를 위해 트랜스파일러를 사용하세요.

In [4]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

### 4. Invoke Sampler and get results

Next, invoke the `run()` method to generate the output. The circuit and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d82863mgbeec73alf9sg


>>> Job Status: QUEUED


In [ ]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(
    f"First ten results for the 'meas' output register: "
    f"{pub_result.data.meas.get_bitstrings()[:10]}"
)

First ten results for the 'meas' output register: ['1100110011001011111111111010000010001010100100011000001011001101000110011000110100100100101010111001110100100000000011111100000', '0101001001010000100111000110110001001101010110110000110111101110001100000001000001111111101110000000010011111100100110001101000', '0111111110011011000011110111010111101100110010001010010001100000000100000000001010101010111010110000001100100001010110000101000', '0000110011001100110011101100000111011001110100001100001100110111010100101010001010000011000111001010101111110110100110001010000', '0011110011100001100110111001000011011111011110111100000110001000111011101101000110011011101011001110110000010010001100100011001', '1010001000010101011100101010101001101000100010011011100110010111010001110111110010100010111010011010110011001101100110010000010', '0001110010001011001100010000000001001101001110101100110011101111100100100110110010101000011010101000101011101011010100000101010', '1110100100001100110010000010011

### 4. Sampler 호출 및 결과 가져오기
다음으로, `run()` 메서드를 호출하여 출력을 생성합니다. Circuit과 선택적 매개변수 값 집합은 *Primitive Unified Bloc* (PUB) 튜플로 입력됩니다.